In [8]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.metrics import silhouette_samples

def compare_clusters_using_means(
    dfs,
    cluster_labels=None,
    alpha=0.05,
    drop_cols=("file_name", "cluster"),
    include_std_n=True,
):
    if cluster_labels is None:
        cluster_labels = [f"c{i}" for i in range(len(dfs))]
    if len(cluster_labels) != len(dfs):
        raise ValueError("cluster_labels must be same length as dfs")

    numeric_features = dfs[0].select_dtypes(include=[np.number]).columns.tolist()
    numeric_features = [f for f in numeric_features if f not in drop_cols]

    rows = []
    for feature in numeric_features:
        feature_means = []
        feature_stds = []
        for df in dfs:
            mean_val = df[feature].dropna().mean()
            std_val = df[feature].dropna().std()
            feature_means.append(mean_val)
            feature_stds.append(std_val)

        arrays = [df[feature].dropna().to_numpy() for df in dfs]

        valid = [a for a in arrays if len(a) > 0]
        if len(valid) < 2:
            continue

        if all(np.nanstd(a) == 0 for a in valid):
            continue

        f_stat, p_value = stats.f_oneway(*valid)

        ns = [int(len(a)) for a in arrays]

        means_arr = np.array(feature_means, dtype=float)
        max_idx = int(np.nanargmax(means_arr))
        min_idx = int(np.nanargmin(means_arr))
        mean_difference = float(means_arr[max_idx] - means_arr[min_idx])

        row = {
            "feature": feature,
            "difference": mean_difference,
            "p_value": float(p_value),
            "significant": bool(p_value < alpha),
            "max_cluster": cluster_labels[max_idx],
            "min_cluster": cluster_labels[min_idx],
        }

        rows.append(row)
        
    # Get rid of the same things like a0_var_spectral_centroid_time_x,20579.209612575687,0.002330426156973986,True,1, a0_var_spectral_centroid_time_y,20579.209612575687,0.002330426156973986,True,1,0
    for row in rows:
        if row["feature"].endswith("_x") or row["feature"].endswith("_y"):
            row["feature"] = row["feature"][:-2]
    seen_features = set()
    unique_rows = []
    for row in rows:
        if row["feature"] not in seen_features:
            unique_rows.append(row)
            seen_features.add(row["feature"])
    
    # order by p_value first, then by difference
    unique_rows.sort(key=lambda x: (x["p_value"], x["difference"]), reverse=True)
    
    out = pd.DataFrame(unique_rows).sort_values(by="p_value", ascending=True)
    return out


def get_silhouette_scores(dfs, cluster_labels=None, drop_cols=("file_name", "cluster")):
    if cluster_labels is None:
        cluster_labels = [f"c{i}" for i in range(len(dfs))]
    if len(cluster_labels) != len(dfs):
        raise ValueError("cluster_labels must be same length as dfs")

    numeric_features = dfs[0].select_dtypes(include=[np.number]).columns.tolist()
    numeric_features = [f for f in numeric_features if f not in drop_cols]

    combined_df = pd.concat(dfs, ignore_index=True)
    combined_df_clean = combined_df.dropna(subset=numeric_features)

    X = combined_df_clean[numeric_features].to_numpy()
    labels = combined_df_clean["cluster"].to_numpy()

    silhouette_vals = silhouette_samples(X, labels)

    combined_df_clean = combined_df_clean.copy()
    combined_df_clean["silhouette_score"] = silhouette_vals
    
    cluster_mean_silhouette = combined_df_clean.groupby("cluster")["silhouette_score"].mean().to_dict()
    combined_df_clean["cluster_mean_silhouette"] = combined_df_clean["cluster"].map(cluster_mean_silhouette)

    return combined_df_clean[["file_name", "cluster", "silhouette_score", "cluster_mean_silhouette"]]


if __name__ == "__main__":
    # Load the dataframes
    df0 = pd.read_csv("../data/3d_umap_clusters/cluster_0.csv")
    df1 = pd.read_csv("../data/3d_umap_clusters/cluster_1.csv")


    dfs = [df0, df1]
    labels = ["0", "1"]

    differences = compare_clusters_using_means(
        dfs,
        cluster_labels=labels,
        alpha=0.05,
        drop_cols=("file_name", "cluster"),
        include_std_n=True,
    )

    out_path = "../visuals/rep_samples/UMAP-HDBSCAN/cluster_differences_with_means.csv"
    differences.to_csv(out_path, index=False)

    out_path_silhouette = "../visuals/rep_samples/UMAP-HDBSCAN/silhouette_scores.csv"
    silhouette_df = get_silhouette_scores(dfs, cluster_labels=labels, drop_cols=("file_name", "cluster"))
    silhouette_df.to_csv(out_path_silhouette, index=False)

    # Show the top 15 most separating features
    print(differences.head(15)[["feature", "difference", "p_value", "max_cluster", "min_cluster"]])
    print(f"\nSaved: {out_path}")

                            feature    difference       p_value max_cluster  \
60               a0_high_band_power  9.367358e-01  2.904789e-80           1   
59    a0_mean_spectral_entropy_time  4.112237e+00  8.159676e-80           1   
58                  a0_crest_factor  5.112926e+00  2.672264e-72           1   
57            a0_iqr_peak_magnitude  2.566888e-03  2.141073e-63           1   
56                 a0_spectral_flux  4.725928e-14  6.638982e-60           1   
55                    a0_burstiness  1.044051e+00  4.351033e-49           1   
54        a0_unused_peak_proportion  7.206258e-01  8.416132e-45           1   
53           a0_skew_peak_magnitude  1.018582e+00  1.195827e-40           1   
52              a0_peaks_per_second  4.044070e+00  2.406275e-39           1   
51                a0_low_band_power  7.728057e-03  6.918543e-37           1   
50             a0_high_to_low_ratio  1.636507e+02  3.224743e-35           1   
49  a0_percent_time_above_threshold  1.826391e-03  9

In [ ]:
# Find the significant features that separate the clusters, and also get silhouette scores for each sample to see how well they fit within their assigned cluster.
def compare_clusters_using_means(
    dfs,
    cluster_labels=None,
    alpha=0.05,
    drop_cols=("file_name", "cluster"),
    include_std_n=True,
):
    if cluster_labels is None:
        cluster_labels = [f"c{i}" for i in range(len(dfs))]
    if len(cluster_labels) != len(dfs):
        raise ValueError("cluster_labels must be same length as dfs")

    numeric_features = dfs[0].select_dtypes(include=[np.number]).columns.tolist()
    numeric_features = [f for f in numeric_features if f not in drop_cols]

    rows = []
    for feature in numeric_features:
        feature_means = []
        feature_stds = []
        for df in dfs:
            mean_val = df[feature].dropna().mean()
            std_val = df[feature].dropna().std()
            feature_means.append(mean_val)
            feature_stds.append(std_val)

        arrays = [df[feature].dropna().to_numpy() for df in dfs]

        valid = [a for a in arrays if len(a) > 0]
        if len(valid) < 2:
            continue

        if all(np.nanstd(a) == 0 for a in valid):
            continue

        f_stat, p_value = stats.f_oneway(*valid)

        ns = [int(len(a)) for a in arrays]

        means_arr = np.array(feature_means, dtype=float)
        max_idx = int(np.nanargmax(means_arr))
        min_idx = int(np.nanargmin(means_arr))
        mean_difference = float(means_arr[max_idx] - means_arr[min_idx])

        row = {
            "feature": feature,
            "difference": mean_difference,
            "p_value": float(p_value),
            "significant": bool(p_value < alpha),
            "max_cluster": cluster_labels[max_idx],
            "min_cluster": cluster_labels[min_idx],
        }

        rows.append(row)
        
    for row in rows:
        if row["feature"].endswith("_x") or row["feature"].endswith("_y"):
            row["feature"] = row["feature"][:-2]
    seen_features = set()
    unique_rows = []
    for row in rows:
        if row["feature"] not in seen_features:
            unique_rows.append(row)
            seen_features.add(row["feature"])
    # order by p_value first, then by difference
    unique_rows.sort(key=lambda x: (x["p_value"], x["difference"]), reverse=True)
    out = pd.DataFrame(unique_rows).sort_values(by="p_value", ascending=True)
    return out

